# 임베딩 모델 3종 비교 — BGE-M3 / KURE-v1 / Qwen3-Embedding-0.6B

**목적**: jo-level(조 단위)과 ho-level(호/목/세목 단위) 두 chunk 셋 각각에 대해,
3개 임베딩 모델의 dense 단일 벡터 brute-force cosine 검색 성능을 gold_standard_v4
(자연어 법률 상담 쿼리 100개, 정답 여러 개인 케이스 포함)로 비교합니다.

**이 실험이 gold_standard_v4를 써도 되는 이유**: gold_standard_v4는 원래 RAG(하이브리드+
리랭커) 평가용으로 만든 셋이지만, 정답 chunk_id는 "이 질의의 법적 정답은 이 조항"이라는
사람의 판단이지 특정 모델의 검색 결과에 의존한 게 아니라서 임베딩 모델 단독 비교에도
그대로 씁니다. 다만 이 실험은 dense 단독이라 실제 하이브리드+리랭커 파이프라인 Recall/MRR
(HoRAG 기준 Recall@10=0.93, MRR=0.7065)보다 낮게 나오는 게 정상입니다 — "임베딩 백본만
떼서 비교"가 목적이라 절대 수치보다 **3개 모델 사이의 상대적 우열**을 봅니다.

**지표**: Recall@1/5/10, MRR(기본) + nDCG@10, MAP@10(다중 정답 랭킹 품질) + 임베딩 차원,
예상 배포 용량(fp16 기준, 실무 배포 관점)

**모델별로 셀을 따로 뒀습니다** — 한 모델이 끝날 때마다 바로 CSV에 저장하기 때문에,
중간에 하나가 OOM 등으로 실패해도 앞서 끝난 모델 결과는 안전합니다. 실패한 모델
셀만 다시 실행하면 됩니다 (다른 모델을 다시 돌릴 필요 없음).

**필요 파일 (아래 셀에서 업로드)**
- `chunks_jo_fixedid.json` (1,186개)
- `chunks_ho_fixedid.json` (7,640개)
- `eval_set.json` (gold_standard_v4, 100개 — 파일명이 다르면 이 이름으로 바꿔서 올려주세요)


## 1. 환경 설정

In [1]:
!pip install -q FlagEmbedding sentence-transformers pandas numpy torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.1 MB/s eta 0:00:00


## 2. 파일 업로드

`chunks_jo_fixedid.json`, `chunks_ho_fixedid.json`, `eval_set.json`(gold_standard_v4)
3개를 한 번에 선택해서 업로드하세요.


In [2]:
from google.colab import files

uploaded = files.upload()


Saving eval_set.json to eval_set.json


In [3]:
from google.colab import files

uploaded = files.upload()

Saving chunks_ho_fixedid.json to chunks_ho_fixedid.json
Saving chunks_jo_fixedid.json to chunks_jo_fixedid.json


## 3. 지표 모듈 작성

In [4]:
%%writefile /content/yoonha_embedding_metrics.py
"""
Workit - 임베딩 모델 성능 평가용 지표 모듈
파일명: yoonha_embedding_metrics.py

Recall@k / MRR 외에 다중 정답(gold_standard_v4엔 정답이 여러 개인 쿼리가
흔함) 랭킹 품질을 정밀하게 보기 위해 nDCG@10, MAP@10을 추가한다.
Recall@k는 "하나라도 top-k에 걸리면 만점" 처리라 다중 정답 케이스에서
"몇 등에 몇 개나 걸렸는지"를 못 잡아낸다 — 그 공백을 nDCG/MAP이 메운다.

전부 이진 관련성(binary relevance) 기준이다 — gold_standard에 정답 간
등급(1순위 정답/2순위 정답 등)이 없어서, "정답이냐 아니냐"만으로 계산한다.
"""

from __future__ import annotations

import math


def recall_at_k(ranked_ids: list[str], gt_set: set[str], k: int) -> float:
    """gt_set 중 하나라도 top-k 안에 있으면 1.0, 아니면 0.0."""
    return 1.0 if gt_set & set(ranked_ids[:k]) else 0.0


def reciprocal_rank(ranked_ids: list[str], gt_set: set[str]) -> float:
    """정답 중 가장 먼저 나오는 위치의 역수. 못 찾으면 0."""
    for rank, cid in enumerate(ranked_ids, 1):
        if cid in gt_set:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(ranked_ids: list[str], gt_set: set[str], k: int) -> float:
    """
    이진 관련성 기준 nDCG@k.
    DCG@k    = sum_{i=1..k} rel(i) / log2(i+1)   (rel(i)는 0 또는 1)
    IDCG@k   = 정답을 전부 상위에 몰아넣었을 때의 이상적 DCG
             = sum_{i=1..min(k,|gt|)} 1 / log2(i+1)
    nDCG@k   = DCG@k / IDCG@k  (IDCG가 0이면 0)
    """
    if not gt_set:
        return 0.0
    dcg = 0.0
    for i, cid in enumerate(ranked_ids[:k], 1):
        if cid in gt_set:
            dcg += 1.0 / math.log2(i + 1)
    ideal_hits = min(len(gt_set), k)
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0


def average_precision_at_k(ranked_ids: list[str], gt_set: set[str], k: int) -> float:
    """
    이진 관련성 기준 AP@k. 분모는 min(|gt|, k)로 정규화해서 0~1 범위로 bound한다
    (k가 |gt|보다 작아서 전부 못 담는 구조적 한계까지 점수에 반영되는 걸 막기 위함
    — 이렇게 안 하면 |gt|가 3인데 k=10이어도 최댓값이 0.3대로 나오는 왜곡이 생김).
    """
    if not gt_set:
        return 0.0
    hits = 0
    precision_sum = 0.0
    for i, cid in enumerate(ranked_ids[:k], 1):
        if cid in gt_set:
            hits += 1
            precision_sum += hits / i
    denom = min(len(gt_set), k)
    return precision_sum / denom if denom > 0 else 0.0


def evaluate_ranking(ranked_ids: list[str], gt_set: set[str], recall_ks: list[int], ndcg_k: int = 10, map_k: int = 10) -> dict:
    """한 쿼리에 대한 전체 지표를 한 번에 계산."""
    result = {f"Recall@{k}": recall_at_k(ranked_ids, gt_set, k) for k in recall_ks}
    result["RR"] = reciprocal_rank(ranked_ids, gt_set)
    result[f"nDCG@{ndcg_k}"] = ndcg_at_k(ranked_ids, gt_set, ndcg_k)
    result[f"MAP@{map_k}"] = average_precision_at_k(ranked_ids, gt_set, map_k)
    return result


Writing /content/yoonha_embedding_metrics.py


## 4. 공통 함수 (전체 모델이 공유)

In [5]:
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from yoonha_embedding_metrics import evaluate_ranking

CHUNKS_JO_PATH = Path("/content/chunks_jo_fixedid.json")
CHUNKS_HO_PATH = Path("/content/chunks_ho_fixedid.json")
GOLD_STANDARD_PATH = Path("/content/eval_set.json")  # gold_standard_v4.json
OUT_PATH = Path("/content/eval_embedding_model_comparison.csv")

RECALL_KS = [1, 5, 10]
NDCG_K = 10
MAP_K = 10

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"디바이스: {DEVICE}")


def load_chunks(path):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    ids = [d["chunk_id"] for d in data]
    texts = [d["text"] for d in data]
    return ids, texts


def load_gold_standard(path=GOLD_STANDARD_PATH):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def get_model_size_mb(model):
    """
    파라미터 수 기준 예상 배포 용량(fp16, 2bytes/param) 추정.
    라이브러리 내부 구조가 버전마다 달라서 흔한 속성 경로를 순서대로 시도하고,
    다 실패하면 None (경고만 출력).
    """
    candidates = [
        lambda m: m,
        lambda m: m.model,
        lambda m: m[0].auto_model,
        lambda m: m.encoder,
    ]
    for get in candidates:
        try:
            inner = get(model)
            n_params = sum(p.numel() for p in inner.parameters())
            if n_params > 0:
                return round(n_params * 2 / (1024 ** 2), 1)
        except Exception:
            continue
    print("  ⚠️  모델 파라미터 수 introspection 실패 — 용량은 공식 스펙에서 수동으로 채워야 함")
    return None


def encode_with_oom_backoff(model, texts, batch_size):
    """
    CUDA OOM이 나면 batch_size를 절반씩 줄여가며 재시도한다 (최소 1까지).
    jo-level처럼 chunk 길이가 들쭉날쭉한 데이터(최대 5,620자)를 긴 시퀀스
    제한이 느슨한 모델(Qwen3-Embedding 등)에 태울 때 특히 필요.
    """
    bs = batch_size
    while True:
        try:
            return np.asarray(model.encode(texts, batch_size=bs, show_progress_bar=True, normalize_embeddings=False))
        except torch.cuda.OutOfMemoryError:
            gc.collect()
            torch.cuda.empty_cache()
            if bs <= 1:
                raise
            bs = max(1, bs // 2)
            print(f"  ⚠️  CUDA OOM 발생 — batch_size를 {bs}로 줄여서 재시도합니다.")


def cosine_topk_all(query_vecs, chunk_vecs, top_k):
    """L2 정규화 후 내적 = cosine similarity. brute-force로 충분히 빠름."""
    q = query_vecs / (np.linalg.norm(query_vecs, axis=1, keepdims=True) + 1e-12)
    c = chunk_vecs / (np.linalg.norm(chunk_vecs, axis=1, keepdims=True) + 1e-12)
    sims = q @ c.T
    top_idx = np.argsort(-sims, axis=1)[:, :top_k]
    return top_idx.tolist()


def evaluate_one_granularity(chunk_ids, top_idx_per_query, gold_standard, gt_field):
    rows = []
    for item, top_idx in zip(gold_standard, top_idx_per_query):
        gt_set = set(item[gt_field])
        ranked_ids = [chunk_ids[i] for i in top_idx]
        rows.append(evaluate_ranking(ranked_ids, gt_set, RECALL_KS, NDCG_K, MAP_K))

    df = pd.DataFrame(rows)
    result = {f"Recall@{k}": round(df[f"Recall@{k}"].mean(), 4) for k in RECALL_KS}
    result["MRR"] = round(df["RR"].mean(), 4)
    result[f"nDCG@{NDCG_K}"] = round(df[f"nDCG@{NDCG_K}"].mean(), 4)
    result[f"MAP@{MAP_K}"] = round(df[f"MAP@{MAP_K}"].mean(), 4)
    return result


def evaluate_model_on_both_granularities(model_name, query_vecs, jo_vecs, ho_vecs, meta):
    """쿼리/jo/ho 임베딩이 이미 계산된 상태에서 두 granularity 지표를 계산하고 바로 저장한다."""
    rows = []
    for granularity, ids, vecs, gt_field in [
        ("jo", jo_ids, jo_vecs, "relevant_docs_jo"),
        ("ho", ho_ids, ho_vecs, "relevant_docs_ho"),
    ]:
        top_idx = cosine_topk_all(query_vecs, vecs, top_k=max(RECALL_KS))
        metrics = evaluate_one_granularity(ids, top_idx, gold_standard, gt_field)
        row = {"model": model_name, "granularity": granularity, **metrics, **meta}
        rows.append(row)
        print(f"  [{granularity}] {metrics}")

    save_rows(rows, [model_name])
    print(f"  💾 저장 완료 ({model_name})")
    return rows


def save_rows(new_rows, model_names_this_run):
    """
    이번에 나온 행을 기존 CSV와 병합해서 즉시 저장한다 (모델별 셀이 끝날 때마다
    호출) — 다른 모델 셀이 나중에 실패해도 이미 저장된 행은 안전하다.
    """
    df_new = pd.DataFrame(new_rows)
    if OUT_PATH.exists():
        df_old = pd.read_csv(OUT_PATH)
        df_old = df_old[~df_old["model"].isin(model_names_this_run)]
        df = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df = df_new
    df.to_csv(OUT_PATH, index=False)
    return df


def free_gpu():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


디바이스: cuda


## 5. 데이터 로드 (한 번만 실행 — 이후 모델별 셀에서 재사용)

In [6]:
jo_ids, jo_texts = load_chunks(CHUNKS_JO_PATH)
ho_ids, ho_texts = load_chunks(CHUNKS_HO_PATH)
gold_standard = load_gold_standard()
queries = [item["query"] for item in gold_standard]

print(f"jo-level {len(jo_ids)}개, ho-level {len(ho_ids)}개, 쿼리 {len(queries)}개 로드 완료")


jo-level 1186개, ho-level 7640개, 쿼리 100개 로드 완료


## 6. BAAI/bge-m3

In [7]:
from FlagEmbedding import BGEM3FlagModel

print("[BAAI/bge-m3] 로드 중...")
_model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=(DEVICE == "cuda"))

def _encode_bge(texts):
    return np.asarray(_model.encode(texts, return_dense=True, return_sparse=False)["dense_vecs"])

_dim = _encode_bge(["dummy"]).shape[1]
_size_mb = get_model_size_mb(_model.model) if hasattr(_model, "model") else None
_meta = {"embedding_dim": _dim, "model_size_mb_fp16": _size_mb}

_jo_vecs = _encode_bge(jo_texts)
_ho_vecs = _encode_bge(ho_texts)
_query_vecs = _encode_bge(queries)

evaluate_model_on_both_granularities("BAAI/bge-m3", _query_vecs, _jo_vecs, _ho_vecs, _meta)

del _model, _jo_vecs, _ho_vecs, _query_vecs
free_gpu()


[BAAI/bge-m3] 로드 중...


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


pre tokenize: 100%|██████████| 5/5 [00:00<00:00,  8.14it/s]

Inference Embeddings: 100%|██████████| 5/5 [00:14<00:00,  2.93s/it]

pre tokenize: 100%|██████████| 30/30 [00:00<00:00, 42.69it/s]

Inference Embeddings: 100%|██████████| 30/30 [00:17<00:00,  1.71it/s]


  [jo] {'Recall@1': np.float64(0.68), 'Recall@5': np.float64(0.88), 'Recall@10': np.float64(0.95), 'MRR': np.float64(0.7785), 'nDCG@10': np.float64(0.7872), 'MAP@10': np.float64(0.7349)}
  [ho] {'Recall@1': np.float64(0.57), 'Recall@5': np.float64(0.78), 'Recall@10': np.float64(0.83), 'MRR': np.float64(0.6546), 'nDCG@10': np.float64(0.6339), 'MAP@10': np.float64(0.5738)}
  💾 저장 완료 (BAAI/bge-m3)


## 7. nlpai-lab/KURE-v1

In [8]:
from sentence_transformers import SentenceTransformer

print("[nlpai-lab/KURE-v1] 로드 중...")
_model = SentenceTransformer("nlpai-lab/KURE-v1", device=DEVICE, trust_remote_code=True)
_model.max_seq_length = 2048  # jo-level 최대 5,620자 chunk 대비 안전 캡

_dim = _model.get_sentence_embedding_dimension()
_size_mb = get_model_size_mb(_model)
_meta = {"embedding_dim": _dim, "model_size_mb_fp16": _size_mb}

_jo_vecs = encode_with_oom_backoff(_model, jo_texts, batch_size=8)   # 긴 텍스트라 작은 배치
_ho_vecs = encode_with_oom_backoff(_model, ho_texts, batch_size=32)
_query_vecs = encode_with_oom_backoff(_model, queries, batch_size=32)

evaluate_model_on_both_granularities("nlpai-lab/KURE-v1", _query_vecs, _jo_vecs, _ho_vecs, _meta)

del _model, _jo_vecs, _ho_vecs, _query_vecs
free_gpu()


[nlpai-lab/KURE-v1] 로드 중...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/16.9k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

/tmp/ipykernel_554/3045502477.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  _dim = _model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/149 [00:00<?, ?it/s]

Batches:   0%|          | 0/239 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

  [jo] {'Recall@1': np.float64(0.71), 'Recall@5': np.float64(0.92), 'Recall@10': np.float64(0.96), 'MRR': np.float64(0.8013), 'nDCG@10': np.float64(0.7991), 'MAP@10': np.float64(0.7415)}
  [ho] {'Recall@1': np.float64(0.54), 'Recall@5': np.float64(0.79), 'Recall@10': np.float64(0.89), 'MRR': np.float64(0.6584), 'nDCG@10': np.float64(0.6492), 'MAP@10': np.float64(0.58)}
  💾 저장 완료 (nlpai-lab/KURE-v1)


## 8. Qwen/Qwen3-Embedding-0.6B

Qwen3-Embedding은 기본적으로 시퀀스 길이 제한이 느슨해서(약 32k) jo-level의
긴 chunk(최대 5,620자)가 큰 배치에 섞이면 attention 메모리가 튀어 CUDA OOM이
날 수 있습니다. `max_seq_length=2048` 캡 + jo-level엔 작은 배치(4) + OOM 시
배치 크기 자동 축소(절반씩, 최소 1까지) 재시도로 대응했습니다.

쿼리에는 공식 권장 instruction 프리픽스를 붙였는데(`QWEN_QUERY_INSTRUCTION`),
예전 구버전 실험과 완전히 동일한 템플릿인지는 확인이 안 된 상태입니다 — 결과가
그때(Recall@1 0.595)와 많이 다르면 이 부분부터 의심해보세요.


In [9]:
from sentence_transformers import SentenceTransformer

QWEN_QUERY_INSTRUCTION = (
    "Instruct: Given a legal consultation query, retrieve the statute article "
    "that provides the relevant legal basis\nQuery: "
)

print("[Qwen/Qwen3-Embedding-0.6B] 로드 중...")
_model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=DEVICE, trust_remote_code=True)
_model.max_seq_length = 2048

_dim = _model.get_sentence_embedding_dimension()
_size_mb = get_model_size_mb(_model)
_meta = {"embedding_dim": _dim, "model_size_mb_fp16": _size_mb}

_jo_vecs = encode_with_oom_backoff(_model, jo_texts, batch_size=4)
_ho_vecs = encode_with_oom_backoff(_model, ho_texts, batch_size=16)
_prefixed_queries = [QWEN_QUERY_INSTRUCTION + q for q in queries]
_query_vecs = encode_with_oom_backoff(_model, _prefixed_queries, batch_size=16)

evaluate_model_on_both_granularities("Qwen/Qwen3-Embedding-0.6B", _query_vecs, _jo_vecs, _ho_vecs, _meta)

del _model, _jo_vecs, _ho_vecs, _query_vecs
free_gpu()


[Qwen/Qwen3-Embedding-0.6B] 로드 중...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

/tmp/ipykernel_554/3139531622.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  _dim = _model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/297 [00:00<?, ?it/s]

Batches:   0%|          | 0/478 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

  [jo] {'Recall@1': np.float64(0.72), 'Recall@5': np.float64(0.94), 'Recall@10': np.float64(0.96), 'MRR': np.float64(0.8129), 'nDCG@10': np.float64(0.8062), 'MAP@10': np.float64(0.7505)}
  [ho] {'Recall@1': np.float64(0.58), 'Recall@5': np.float64(0.8), 'Recall@10': np.float64(0.87), 'MRR': np.float64(0.6795), 'nDCG@10': np.float64(0.657), 'MAP@10': np.float64(0.5951)}
  💾 저장 완료 (Qwen/Qwen3-Embedding-0.6B)


## 9. 결과 확인 + 다운로드

In [10]:
df = pd.read_csv(OUT_PATH)

print("=== jo-level (조 단위) ===")
display(df[df["granularity"] == "jo"].drop(columns=["granularity"]))

print("\n=== ho-level (호/목/세목 단위) ===")
display(df[df["granularity"] == "ho"].drop(columns=["granularity"]))


=== jo-level (조 단위) ===


,model,Recall@1,Recall@5,Recall@10,MRR,nDCG@10,MAP@10,embedding_dim,model_size_mb_fp16
0,BAAI/bge-m3,0.68,0.88,0.95,0.7785,0.7872,0.7349,1024,1084.9
2,nlpai-lab/KURE-v1,0.71,0.92,0.96,0.8013,0.7991,0.7415,1024,1082.9
4,Qwen/Qwen3-Embedding-0.6B,0.72,0.94,0.96,0.8129,0.8062,0.7505,1024,1136.4



=== ho-level (호/목/세목 단위) ===


,model,Recall@1,Recall@5,Recall@10,MRR,nDCG@10,MAP@10,embedding_dim,model_size_mb_fp16
1,BAAI/bge-m3,0.57,0.78,0.83,0.6546,0.6339,0.5738,1024,1084.9
3,nlpai-lab/KURE-v1,0.54,0.79,0.89,0.6584,0.6492,0.5800,1024,1082.9
5,Qwen/Qwen3-Embedding-0.6B,0.58,0.80,0.87,0.6795,0.6570,0.5951,1024,1136.4


In [11]:
from google.colab import files

files.download(str(OUT_PATH))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>